# Independent Lab Work No. 1 — Web Scraping & Descriptive Data Analysis

**Website:** [books.toscrape.com](https://books.toscrape.com/)  
**Tool:** BeautifulSoup + Requests  
**Goal:** Scrape 500 books, save as CSV, and perform descriptive statistics with visualizations.

---
## Step 1 — Import Libraries

We import:
- **requests** — to send HTTP requests and download web pages.
- **BeautifulSoup** (bs4) — to parse HTML and extract data from the page.
- **pandas** — to organize scraped data into a DataFrame and perform analysis.
- **matplotlib / seaborn** — to create visualizations (box plot, bar chart, scatter plot).
- **concurrent.futures** — to fetch many detail pages in parallel (much faster than sequential).
- **time** — to measure how long the scraping takes.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('All libraries imported successfully!')

---
## Step 2 — Web Scraping from books.toscrape.com

### How the scraper works

The scraper runs in **two phases** for speed:

**Phase 1 — Catalogue pages (25 pages, sequential)**
1. For each of the 25 catalogue pages we download the HTML with `requests.get()`.
2. Parse it with `BeautifulSoup` and find every `<article class="product_pod">`.
3. From each book card extract: **title**, **price (£)**, **rating (1–5)**, **availability**, and the **link to the detail page**.

**Phase 2 — Detail pages (500 pages, parallel with 10 threads)**
1. Using `ThreadPoolExecutor` we fetch all 500 book detail pages concurrently.
2. From each detail page we extract: **category** (breadcrumb), **description**, **UPC**, **tax**, and **number of reviews**.

**Phase 3 — Merge** the catalogue data with the detail data into a final list.

In [ ]:
BASE_URL = 'https://books.toscrape.com/catalogue/'
START_URL = 'https://books.toscrape.com/catalogue/page-{}.html'
MAX_PAGES = 25  # 25 pages × 20 books = 500 books

# Map word-ratings to numbers
RATING_MAP = {
    'One': 1,
    'Two': 2,
    'Three': 3,
    'Four': 4,
    'Five': 5
}

# Reusable session for faster HTTP (connection pooling)
session = requests.Session()


def get_soup(url: str) -> BeautifulSoup:
    """Download a page and return a BeautifulSoup object."""
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return BeautifulSoup(response.text, 'lxml')


def scrape_book_detail(detail_href: str) -> dict:
    """Visit a single book's detail page and extract extra info."""
    url = BASE_URL + detail_href
    soup = get_soup(url)

    # Category from breadcrumb:  Home > Books > <Category> > <Title>
    breadcrumb = soup.select('ul.breadcrumb li')
    category = breadcrumb[2].get_text(strip=True) if len(breadcrumb) >= 3 else None

    # Description
    desc_div = soup.select_one('#product_description')
    description = desc_div.find_next_sibling('p').get_text(strip=True) if desc_div else None

    # Product information table
    table = {}
    for row in soup.select('table.table-striped tr'):
        key = row.select_one('th').get_text(strip=True)
        val = row.select_one('td').get_text(strip=True)
        table[key] = val

    upc = table.get('UPC')
    tax = table.get('Tax', '£0.00').replace('£', '').replace('Â', '')
    num_reviews = table.get('Number of reviews', '0')

    return {
        'category': category,
        'description': description,
        'upc': upc,
        'tax': float(tax),
        'num_reviews': int(num_reviews)
    }


start_time = time.time()

# ── Phase 1: collect basic info from 25 catalogue pages ───────────────
print('Phase 1 — Scraping catalogue pages …')
catalogue_books = []

for page in range(1, MAX_PAGES + 1):
    soup = get_soup(START_URL.format(page))
    for article in soup.select('article.product_pod'):
        title_tag = article.select_one('h3 a')
        price_text = article.select_one('p.price_color').get_text(strip=True)
        rating_cls = article.select_one('p.star-rating')['class']

        catalogue_books.append({
            'title': title_tag['title'],
            'detail_href': title_tag['href'],
            'price': float(price_text.replace('£', '').replace('Â', '')),
            'rating': RATING_MAP.get(rating_cls[1], 0),
            'availability': article.select_one('p.availability').get_text(strip=True),
        })
    if page % 5 == 0:
        print(f'  … page {page}/{MAX_PAGES}  (books so far: {len(catalogue_books)})')

print(f'  Catalogue done — {len(catalogue_books)} books found.\n')

# ── Phase 2: fetch detail pages in parallel (10 threads) ─────────────
print('Phase 2 — Fetching detail pages in parallel …')
details_map = {}  # detail_href -> detail dict


def _fetch_detail(href):
    return href, scrape_book_detail(href)


with ThreadPoolExecutor(max_workers=10) as pool:
    futures = {pool.submit(_fetch_detail, b['detail_href']): b for b in catalogue_books}
    done = 0
    for future in as_completed(futures):
        href, detail = future.result()
        details_map[href] = detail
        done += 1
        if done % 100 == 0:
            print(f'  … {done}/{len(catalogue_books)} detail pages fetched')

print(f'  All {len(details_map)} detail pages fetched.\n')

# ── Phase 3: merge catalogue + detail data ────────────────────────────
books = []
for b in catalogue_books:
    d = details_map[b['detail_href']]
    books.append({
        'title': b['title'],
        'price': b['price'],
        'rating': b['rating'],
        'availability': b['availability'],
        'category': d['category'],
        'description': d['description'],
        'upc': d['upc'],
        'tax': d['tax'],
        'num_reviews': d['num_reviews'],
    })

elapsed = time.time() - start_time
print(f'Done!  Total books scraped: {len(books)}  ({elapsed:.1f} seconds)')

---
## Step 3 — Save the Dataset to CSV

We convert the list of dictionaries into a **pandas DataFrame** and save it as `books_dataset.csv`.  
This structured format makes it easy to reload later without re-scraping.

In [ ]:
df = pd.DataFrame(books)
df.to_csv('books_dataset.csv', index=False)
print(f'Dataset saved to books_dataset.csv  ({df.shape[0]} rows, {df.shape[1]} columns)')
df.head(10)

---
## Step 4 — Load & Explore the Dataset

We reload the CSV (to prove the file works) and do a quick exploration:
- `.shape` — number of rows and columns.
- `.dtypes` — data types of each column.
- `.info()` — non-null counts (helps spot missing values early).
- `.head()` — first few rows for a sanity check.

In [ ]:
df = pd.read_csv('books_dataset.csv')
print(f'Shape: {df.shape}\n')
print('--- Data Types ---')
print(df.dtypes)
print('\n--- Info ---')
df.info()
print('\n--- First 5 Rows ---')
df.head()

---
## Step 5 — Descriptive Statistics

We compute key statistics for the numerical columns (`price`, `rating`, `tax`, `num_reviews`):

| Statistic | What it tells us |
|-----------|------------------|
| **Mean** | Average value — the central tendency |
| **Median** | Middle value — robust to outliers |
| **Mode** | Most frequent value |
| **Std** | Standard deviation — spread around the mean |
| **Min / Max** | Range of the data |
| **25% / 75%** | Quartiles — helps detect skewness |

In [ ]:
# Built-in describe gives count, mean, std, min, 25%, 50% (median), 75%, max
num_cols = ['price', 'rating', 'tax', 'num_reviews']
print('=== describe() for numerical columns ===')
print(df[num_cols].describe())

# Add mode explicitly (describe does not include it)
print('\n=== Mode ===')
for col in num_cols:
    mode_val = df[col].mode()[0]
    print(f'  {col}: {mode_val}')

# Summary table
print('\n=== Custom Summary Table ===')
summary = pd.DataFrame({
    'mean': df[num_cols].mean(),
    'median': df[num_cols].median(),
    'mode': df[num_cols].mode().iloc[0],
    'std': df[num_cols].std(),
    'min': df[num_cols].min(),
    'max': df[num_cols].max(),
})
summary

---
## Step 6 — Identify and Handle Missing Values

1. **Identify** — count `NaN`s per column with `.isnull().sum()`.
2. **Handle** — we use two strategies:
   - **Deletion**: drop rows where critical fields (title, price) are missing.
   - **Imputation**: fill missing `description` with `"No description available"`; fill missing numeric values with the column median.

In [ ]:
print('=== Missing Values Before Handling ===')
missing_before = df.isnull().sum()
print(missing_before[missing_before > 0] if missing_before.sum() > 0 else 'No missing values found!')
print(f'Total missing cells: {missing_before.sum()}')

# --- Deletion: drop rows where title or price is missing ---
rows_before = len(df)
df.dropna(subset=['title', 'price'], inplace=True)
print(f'\nRows dropped (missing title/price): {rows_before - len(df)}')

# --- Imputation ---
# Fill missing descriptions with a placeholder string
df['description'] = df['description'].fillna('No description available')

# Fill any remaining numeric NaNs with the column median
for col in ['price', 'rating', 'tax', 'num_reviews']:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f'  Imputed {col} NaNs with median = {median_val}')

print('\n=== Missing Values After Handling ===')
missing_after = df.isnull().sum()
print(missing_after[missing_after > 0] if missing_after.sum() > 0 else 'No missing values remain!')
print(f'Final dataset shape: {df.shape}')

---
## Step 7 — Visualizations

We create three required plots:

1. **Box Plot** — shows the distribution of `price` (median, quartiles, outliers).
2. **Bar Chart** — shows how many books belong to each `category` (categorical distribution).
3. **Scatter Plot** — explores the relationship between `price` and `rating` (two numerical variables).

### 7.1 — Box Plot: Distribution of Book Prices

A box plot visualizes:
- The **median** (line inside the box).
- The **interquartile range (IQR)** (the box itself — 25th to 75th percentile).
- **Whiskers** extending to 1.5 × IQR.
- **Outliers** as individual dots beyond the whiskers.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='price', color='#5B9BD5', width=0.4, ax=ax)
ax.set_title('Distribution of Book Prices (£)', fontsize=16, fontweight='bold')
ax.set_xlabel('Price (£)', fontsize=13)
plt.tight_layout()
plt.savefig('boxplot_prices.png', dpi=150)
plt.show()

### 7.2 — Bar Chart: Number of Books per Category

This bar chart shows the **frequency of each category** — i.e., how many books belong to each genre.  
It helps identify which genres are over- or under-represented on the website.

In [ ]:
cat_counts = df['category'].value_counts()

fig, ax = plt.subplots(figsize=(14, 7))
sns.barplot(x=cat_counts.values, y=cat_counts.index, palette='viridis', ax=ax)
ax.set_title('Number of Books per Category', fontsize=16, fontweight='bold')
ax.set_xlabel('Count', fontsize=13)
ax.set_ylabel('Category', fontsize=13)
plt.tight_layout()
plt.savefig('barchart_categories.png', dpi=150)
plt.show()

### 7.3 — Scatter Plot: Price vs. Rating

A scatter plot reveals whether there is a **relationship** between two numerical variables.  
Here we check: *Do higher-rated books tend to cost more (or less)?*  
We add a small jitter to the rating axis because ratings are discrete (1–5) and points would overlap.

In [ ]:
# Add jitter to rating so dots don't overlap on the discrete 1-5 scale
jitter = np.random.uniform(-0.2, 0.2, size=len(df))

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    df['rating'] + jitter,
    df['price'],
    alpha=0.5,
    edgecolors='white',
    linewidth=0.5,
    s=50,
    c=df['price'],
    cmap='plasma'
)
ax.set_title('Book Price vs. Rating', fontsize=16, fontweight='bold')
ax.set_xlabel('Rating (1–5, jittered)', fontsize=13)
ax.set_ylabel('Price (£)', fontsize=13)
ax.set_xticks([1, 2, 3, 4, 5])
plt.tight_layout()
plt.savefig('scatter_price_rating.png', dpi=150)
plt.show()

---
## Summary

| Step | What we did |
|------|-------------|
| 1 | Imported libraries (requests, BeautifulSoup, pandas, seaborn, matplotlib) |
| 2 | Scraped **500 books** (25 pages) from books.toscrape.com — titles, prices, ratings, categories, descriptions, UPC, tax, reviews |
| 3 | Saved the dataset to `books_dataset.csv` |
| 4 | Loaded the CSV and explored data types, shape, and first rows |
| 5 | Computed descriptive statistics: mean, median, mode, std, min, max, quartiles |
| 6 | Identified missing values and handled them (deletion + imputation) |
| 7 | Created three visualizations: box plot (prices), bar chart (categories), scatter plot (price vs. rating) |